In [6]:
import os

def load_agent_positions(save_dir: str, start_index: int, last_index: int):
    """Load agent positions saved as `{index}_agent_positions.npz`.

    Returns a dict with keys `agent_xy` (concatenated numpy array) and `update_idx`.
    """
    import numpy as _np
    import re

    files = []
    for name in os.listdir(save_dir):
        match = re.match(r"^(\d+)_agent_positions\.npz$", name)
        if match:
            files.append((int(match.group(1)), os.path.join(save_dir, name)))

    if not files:
        raise FileNotFoundError(f"No '*_agent_positions.npz' files found in {save_dir}")

    files.sort(key=lambda x: x[0])
    candidates = files[start_index:last_index]

    if not candidates:
        raise FileNotFoundError("No candidate files available")

    agent_xy_list = []
    update_indices = []

    for selected_idx, fname in candidates:
        with _np.load(fname) as data:
            agent_xy_list.append(data["agent_xy"].copy())
            update_indices.append(int(data["update_idx"]) if "update_idx" in data else selected_idx)

    agent_xy = _np.concatenate(agent_xy_list, axis=0)

    return {"agent_xy": agent_xy, "update_idx": update_indices}


import imageio.v2 as imageio


def _frames_to_mp4(frames, output_path, duration: float = 0.5):
    """Write RGB frames to an mp4 file via imageio/ffmpeg.

    If frame sizes differ, resize all frames to match the last image.
    """
    import numpy as _np
    from PIL import Image

    fps = 1.0 / duration if duration and duration > 0 else 2.0
    target_h, target_w = frames[-1].shape[:2]

    rgb_frames = []
    for img in frames:
        if img.ndim == 3 and img.shape[-1] == 4:
            img = img[..., :3]
        if img.shape[0] != target_h or img.shape[1] != target_w:
            pil_img = Image.fromarray(img)
            pil_img = pil_img.resize((target_w, target_h), Image.Resampling.LANCZOS)
            img = _np.asarray(pil_img)
        rgb_frames.append(img)
    imageio.mimsave(output_path, rgb_frames, fps=fps)


def _checkpoint_color(uidx, step_min, step_max, cmap):
    import numpy as _np

    if step_max > step_min:
        t = float((uidx - step_min) / (step_max - step_min))
    else:
        t = 0.0
    return _np.asarray(cmap(t), dtype=_np.float32)


def _rasterize_checkpoint_layer(xy, color_rgba, xlim, ylim, size, point_alpha=0.1):
    import numpy as _np

    width, height = size
    layer = _np.zeros((height, width, 4), dtype=_np.float32)
    if xy.shape[0] == 0:
        return layer

    x_span = xlim[1] - xlim[0]
    y_span = ylim[1] - ylim[0]
    if x_span <= 0 or y_span <= 0:
        return layer

    cols = _np.clip(
        ((xy[:, 0] - xlim[0]) / x_span * (width - 1)).astype(_np.int32),
        0,
        width - 1,
    )
    rows = _np.clip(
        ((ylim[1] - xy[:, 1]) / y_span * (height - 1)).astype(_np.int32),
        0,
        height - 1,
    )

    r, g, b, _ = color_rgba
    np_add = _np.add.at
    np_add(layer[..., 0], (rows, cols), r * point_alpha)
    np_add(layer[..., 1], (rows, cols), g * point_alpha)
    np_add(layer[..., 2], (rows, cols), b * point_alpha)
    np_add(layer[..., 3], (rows, cols), point_alpha)
    return layer


def _alpha_over(dst, src):
    import numpy as _np

    src_a = src[..., 3:4]
    dst_a = dst[..., 3:4]
    out_a = src_a + dst_a * (1.0 - src_a)
    out_rgb = _np.zeros_like(dst[..., :3])
    mask = out_a[..., 0] > 0
    out_rgb[mask] = (
        src[..., :3][mask] * src_a[mask]
        + dst[..., :3][mask] * dst_a[mask] * (1.0 - src_a[mask])
    ) / out_a[mask]
    result = _np.zeros_like(dst)
    result[..., :3] = out_rgb
    result[..., 3:4] = out_a
    return result


def _composite_layers(layers):
    import numpy as _np

    canvas = _np.zeros(layers[0].shape, dtype=_np.float32)
    for layer in layers:
        canvas = _alpha_over(canvas, layer)
    return canvas


def _layer_frame_to_rgb(frame_rgba):
    import numpy as _np

    alpha = frame_rgba[..., 3:4]
    rgb = frame_rgba[..., :3] * alpha + (1.0 - alpha)
    return _np.clip(rgb * 255.0, 0, 255).astype(_np.uint8)


def _add_frame_title(rgb_frame, title):
    import numpy as _np
    from PIL import Image, ImageDraw

    img = Image.fromarray(rgb_frame)
    draw = ImageDraw.Draw(img)
    draw.text((8, 8), title, fill=(0, 0, 0))
    return _np.asarray(img)


def load_agent_positions_visuals(
    save_dir: str,
    start_index: int,
    last_index: int,
    output_path=None,
    duration: float = 0.5,
    render_mode: str = "layers",
    max_points_per_checkpoint=None,
    image_size=(800, 800),
    dpi: int = 150,
    store_frames: bool = True,
):
    """Load agent position checkpoints and create a cumulative scatter MP4 animation.

    Expects files named `{update_idx}_agent_positions.npz`, sorted by update index.
    Selects the slice `[start_index:last_index]` of that sorted list. Each frame
    cumulatively adds the next checkpoint's points so early training steps appear
    first and the scatter trail grows over time.

    By default uses fast checkpoint layer compositing. Set `render_mode='scatter'`
    to use the slower matplotlib scatter fallback.

    Returns a dict with keys `video_path`, `update_idx`, and `frames`.
    """
    import io
    import re

    import matplotlib
    import numpy as _np

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    files = []
    for name in os.listdir(save_dir):
        match = re.match(r"^(\d+)_agent_positions\.npz$", name)
        if match:
            files.append((int(match.group(1)), os.path.join(save_dir, name)))

    if not files:
        raise FileNotFoundError(f"No '*_agent_positions.npz' files found in {save_dir}")

    files.sort(key=lambda x: x[0])
    candidates = files[start_index:last_index]

    if not candidates:
        raise FileNotFoundError("No candidate files available")

    checkpoints = []
    update_indices = []
    for selected_idx, fname in candidates:
        with _np.load(fname) as data:
            xy = data["agent_xy"].copy().reshape(-1, 2)
            uidx = int(data["update_idx"]) if "update_idx" in data else selected_idx
            if (
                max_points_per_checkpoint is not None
                and xy.shape[0] > max_points_per_checkpoint
            ):
                pick = _np.random.choice(
                    xy.shape[0], int(max_points_per_checkpoint), replace=False
                )
                xy = xy[pick]
            checkpoints.append((xy, uidx))
            update_indices.append(uidx)

    all_xy = _np.concatenate([xy for xy, _ in checkpoints], axis=0)
    if all_xy.shape[0] == 0:
        raise ValueError("Need at least one trajectory point to plot.")

    padding = 0.05
    x_min, x_max = float(all_xy[:, 0].min()), float(all_xy[:, 0].max())
    y_min, y_max = float(all_xy[:, 1].min()), float(all_xy[:, 1].max())
    x_pad = (x_max - x_min) * padding if x_max > x_min else 1.0
    y_pad = (y_max - y_min) * padding if y_max > y_min else 1.0
    xlim = (x_min - x_pad, x_max + x_pad)
    ylim = (y_min - y_pad, y_max + y_pad)

    step_min = float(min(uidx for _, uidx in checkpoints))
    step_max = float(max(uidx for _, uidx in checkpoints))

    cmap = plt.get_cmap("Blues")
    mp4_frames = []
    returned_frames = [] if store_frames else None

    if render_mode == "layers":
        width, height = image_size
        canvas = _np.zeros((height, width, 4), dtype=_np.float32)
        for xy, uidx in checkpoints:
            color = _checkpoint_color(uidx, step_min, step_max, cmap)
            layer = _rasterize_checkpoint_layer(
                xy, color, xlim, ylim, (width, height), point_alpha=0.1
            )
            canvas = _alpha_over(canvas, layer)
            rgb = _layer_frame_to_rgb(canvas)
            rgb = _add_frame_title(rgb, f"Agent Positions (update {uidx})")
            mp4_frames.append(rgb)
            if store_frames:
                returned_frames.append(rgb)
    elif render_mode == "scatter":
        cumulative_xy = []
        cumulative_steps = []
        for xy, uidx in checkpoints:
            cumulative_xy.append(xy)
            cumulative_steps.append(_np.full(xy.shape[0], uidx, dtype=_np.float64))

            xy_cat = _np.concatenate(cumulative_xy, axis=0)
            steps_cat = _np.concatenate(cumulative_steps, axis=0)

            if step_max > step_min:
                color_values = (steps_cat - step_min) / (step_max - step_min)
            else:
                color_values = _np.zeros_like(steps_cat, dtype=_np.float64)

            fig, ax = plt.subplots(figsize=(6.5, 6))
            ax.scatter(
                xy_cat[:, 0],
                xy_cat[:, 1],
                c=cmap(color_values),
                s=1,
                alpha=0.1,
                linewidths=0,
            )
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            ax.set_title(f"Agent Positions (update {uidx})")
            ax.axis("off")

            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
            plt.close(fig)
            buf.seek(0)
            rgb = imageio.imread(buf)
            if rgb.ndim == 3 and rgb.shape[-1] == 4:
                rgb = rgb[..., :3]
            mp4_frames.append(rgb)
            if store_frames:
                returned_frames.append(rgb)
    else:
        raise ValueError("render_mode must be 'layers' or 'scatter'")

    if output_path is None:
        output_path = os.path.join(
            save_dir, f"agent_positions_visuals_{start_index}_{last_index}.mp4"
        )

    _frames_to_mp4(mp4_frames, output_path, duration=duration)
    print(f"Saved agent positions visuals to {output_path}")

    return {
        "video_path": output_path,
        "update_idx": update_indices,
        "frames": returned_frames if store_frames else [],
    }


def load_teacher_goal_visuals(
    save_dir: str,
    start_index: int,
    last_index: int,
    output_path=None,
    duration: float = 0.5,
):
    """Load teacher goal count images and create an MP4 animation.

    Expects files named `..._teacher_goal_counts_{update_idx}.png`, sorted by
    update index. Selects the slice `[start_index:last_index]` of that sorted
    list and writes an animation with imageio.

    Returns a dict with keys `video_path`, `update_idx`, and `frames`.
    """
    import re

    files = []
    # print(save_dir)
    for name in os.listdir(save_dir):
        # print(name)
        # purejaxrl_ppo_brax_ant_u_maze_single_goal_teacher_goal_counts_0.png
        match = re.match(
            r"^purejaxrl_ppo_brax_ant_u_maze_single_goal_teacher_goal_counts_(\d+)\.png$",
            name,
        )
        if match:
            files.append((int(match.group(1)), os.path.join(save_dir, name)))

    if not files:
        raise FileNotFoundError(
            f"No '*_teacher_goal_counts_*.png' files found in {save_dir}"
        )

    files.sort(key=lambda x: x[0])
    candidates = files[start_index:last_index]

    if not candidates:
        raise FileNotFoundError("No candidate files available")

    update_indices = []
    frames = []
    for selected_idx, fname in candidates:
        frames.append(imageio.imread(fname))
        update_indices.append(selected_idx)

    if output_path is None:
        output_path = os.path.join(
            save_dir, f"teacher_goal_visuals_{start_index}_{last_index}.mp4"
        )

    _frames_to_mp4(frames, output_path, duration=duration)
    print(f"Saved teacher goal visuals to {output_path}")

    return {
        "video_path": output_path,
        "update_idx": update_indices,
        "frames": frames,
    }


def load_empowerment_visuals(
    save_dir: str,
    start_index: int,
    last_index: int,
    output_path=None,
    duration: float = 0.5,
):
    """Load teacher empowerment images and create an MP4 animation.

    Expects files named `..._teacher_empowerment_grid_{update_idx}.png`, sorted
    by update index. Selects the slice `[start_index:last_index]` of that sorted
    list and writes an animation with imageio.

    Returns a dict with keys `video_path`, `update_idx`, and `frames`.
    """
    import re

    files = []
    for name in os.listdir(save_dir):
        # purejaxrl_ppo_brax_ant_u_maze_single_goal_teacher_empowerment_grid_0.png
        match = re.match(
            r"^purejaxrl_ppo_brax_ant_u_maze_single_goal_teacher_empowerment_grid_(\d+)\.png$",
            name,
        )
        if match:
            files.append((int(match.group(1)), os.path.join(save_dir, name)))

    if not files:
        raise FileNotFoundError(
            f"No '*_teacher_empowerment_grid_*.png' files found in {save_dir}"
        )

    files.sort(key=lambda x: x[0])
    candidates = files[start_index:last_index]

    if not candidates:
        raise FileNotFoundError("No candidate files available")

    update_indices = []
    frames = []
    for selected_idx, fname in candidates:
        frames.append(imageio.imread(fname))
        update_indices.append(selected_idx)

    if output_path is None:
        output_path = os.path.join(
            save_dir, f"teacher_empowerment_visuals_{start_index}_{last_index}.mp4"
        )

    _frames_to_mp4(frames, output_path, duration=duration)
    print(f"Saved teacher empowerment visuals to {output_path}")

    return {
        "video_path": output_path,
        "update_idx": update_indices,
        "frames": frames,
    }


def load_lp_visuals(
    save_dir: str,
    start_index: int,
    last_index: int,
    output_path=None,
    duration: float = 0.5,
):
    """Load teacher empowerment images and create an MP4 animation.

    Expects files named `..._teacher_empowerment_grid_{update_idx}.png`, sorted
    by update index. Selects the slice `[start_index:last_index]` of that sorted
    list and writes an animation with imageio.

    Returns a dict with keys `video_path`, `update_idx`, and `frames`.
    """
    import re

    files = []
    for name in os.listdir(save_dir):
        # purejaxrl_ppo_brax_ant_u_maze_single_goal_teacher_empowerment_grid_0.png
        match = re.match(
            r"^purejaxrl_ppo_brax_ant_u_maze_single_goal_teacher_learning_progress_(\d+)\.png$",
            name,
        )
        if match:
            files.append((int(match.group(1)), os.path.join(save_dir, name)))

    if not files:
        raise FileNotFoundError(
            f"No '*_teacher_empowerment_grid_*.png' files found in {save_dir}"
        )

    files.sort(key=lambda x: x[0])
    candidates = files[start_index:last_index]

    if not candidates:
        raise FileNotFoundError("No candidate files available")

    update_indices = []
    frames = []
    for selected_idx, fname in candidates:
        frames.append(imageio.imread(fname))
        update_indices.append(selected_idx)

    if output_path is None:
        output_path = os.path.join(
            save_dir, f"teacher_lp_visuals_{start_index}_{last_index}.mp4"
        )

    _frames_to_mp4(frames, output_path, duration=duration)
    print(f"Saved teacher lp visuals to {output_path}")

    return {
        "video_path": output_path,
        "update_idx": update_indices,
        "frames": frames,
    }


def _maze_env_for_name(maze_name):
    """Resolve ``maze_name`` to ``(module, env_class, layout_name)``.

    ``ant_*`` and a bare layout name use ``envs.ant_maze``. ``humanoid_*`` uses
    ``envs.humanoid_maze``. The layout name is the suffix after the env prefix.
    """
    import sys

    here = os.getcwd()
    for _ in range(4):
        purejaxrl_dir = os.path.join(here, "purejaxrl")
        if os.path.isdir(os.path.join(purejaxrl_dir, "envs")):
            if purejaxrl_dir not in sys.path:
                sys.path.insert(0, purejaxrl_dir)
            break
        if os.path.basename(here) == "purejaxrl" and os.path.isdir(
            os.path.join(here, "envs")
        ):
            if here not in sys.path:
                sys.path.insert(0, here)
            break
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent

    if maze_name.startswith("humanoid_"):
        import envs.humanoid_maze as maze_module

        env_cls = maze_module.HumanoidMaze
        layout_name = maze_name[len("humanoid_") :]
    else:
        import envs.ant_maze as maze_module

        env_cls = maze_module.AntMaze
        layout_name = maze_name[len("ant_") :] if maze_name.startswith("ant_") else maze_name

    if not hasattr(maze_module, layout_name.upper()):
        raise ValueError(
            f"Unknown maze layout {layout_name!r} for maze name {maze_name!r}"
        )
    return maze_module, env_cls, layout_name


def maze_wall_segments(maze_name):
    """Wall fills, outer line frame, starts/goals, and axis limits.

    Scale is the env class default ``maze_size_scaling``. Each wall cell is a
    box centered at ``(i * scale, j * scale)`` with half-size ``0.5 * scale``,
    matching ``make_maze``. All wall cells (including the outer frame) are
    filled rectangles so layouts like U-maze keep their corridor shape. A black
    rectangular outline is also returned for a crisp outer border. Start (``R``)
    and goal (``G``) cell centers are returned as ``(N, 2)`` arrays.
    """
    import inspect

    import numpy as _np

    maze_module, env_cls, layout_name = _maze_env_for_name(maze_name)
    layout = getattr(maze_module, layout_name.upper())
    reset_token = maze_module.RESET
    goal_token = maze_module.GOAL
    scale = float(
        inspect.signature(env_cls.__init__).parameters["maze_size_scaling"].default
    )
    half = 0.5 * scale
    n_rows = len(layout)
    n_cols = len(layout[0])

    fills = []
    starts = []
    goals = []
    for i, row in enumerate(layout):
        for j, cell in enumerate(row):
            cx = i * scale
            cy = j * scale
            if cell == reset_token:
                starts.append((cx, cy))
            elif cell == goal_token:
                goals.append((cx, cy))
            if cell != 1:
                continue
            fills.append((cx - half, cy - half, scale, scale))

    x0, x1 = -half, (n_rows - 1) * scale + half
    y0, y1 = -half, (n_cols - 1) * scale + half
    segments = _np.asarray(
        [
            [(x0, y0), (x1, y0)],
            [(x1, y0), (x1, y1)],
            [(x1, y1), (x0, y1)],
            [(x0, y1), (x0, y0)],
        ],
        dtype=_np.float64,
    )
    fills = (
        _np.asarray(fills, dtype=_np.float64)
        if fills
        else _np.zeros((0, 4), dtype=_np.float64)
    )
    starts = (
        _np.asarray(starts, dtype=_np.float64)
        if starts
        else _np.zeros((0, 2), dtype=_np.float64)
    )
    goals = (
        _np.asarray(goals, dtype=_np.float64)
        if goals
        else _np.zeros((0, 2), dtype=_np.float64)
    )
    return segments, fills, starts, goals, (x0, x1), (y0, y1)


def _load_env_steps_per_update(save_dir):
    """Read ``NUM_STEPS * NUM_ENVS`` from ``config.json`` next to ``save_dir``."""
    import json

    config_path = os.path.join(os.path.dirname(save_dir), "config.json")
    if not os.path.isfile(config_path):
        config_path = os.path.join(save_dir, "config.json")
    if not os.path.isfile(config_path):
        raise FileNotFoundError(
            f"Could not find config.json near {save_dir} to convert update "
            "indices to environment steps. Pass num_steps and num_envs."
        )
    with open(config_path) as f:
        config = json.load(f)
    return int(config["NUM_STEPS"]) * int(config["NUM_ENVS"])


def _early_dense_indices(n_files, n_snapshots, power=2.0):
    """Checkpoint indices denser early in training (``t ** power`` spacing)."""
    import numpy as _np

    if n_snapshots <= 1:
        return _np.array([0], dtype=_np.int32)
    n_snapshots = min(int(n_snapshots), int(n_files))
    fracs = _np.linspace(0.0, 1.0, n_snapshots) ** float(power)
    idxs = _np.round(fracs * (n_files - 1)).astype(_np.int32)
    idxs[0] = 0
    idxs[-1] = n_files - 1
    for i in range(1, len(idxs)):
        if idxs[i] <= idxs[i - 1]:
            idxs[i] = idxs[i - 1] + 1
    if idxs[-1] > n_files - 1:
        idxs[-1] = n_files - 1
        for i in range(len(idxs) - 2, -1, -1):
            if idxs[i] >= idxs[i + 1]:
                idxs[i] = idxs[i + 1] - 1
        idxs[0] = 0
    return idxs


def plot_coverage_snapshots(
    save_dir,
    maze_name,
    output_path=None,
    n_snapshots=4,
    num_steps=None,
    num_envs=None,
    dpi=250,
    early_spacing_power=2.0,
    font_scale=1.0,
):
    """Scatter agent (x, y) coverage at training snapshots denser early on.

    Loads ``{update_idx}_agent_positions.npz`` from ``save_dir``, selects
    ``n_snapshots`` checkpoints from the first (t=0) through the last with
    ``early_spacing_power`` (values > 1 pack more panels early), and draws each
    snapshot window on its own panel with maze walls. A shared Environment-steps
    arrow under the panels marks each snapshot's env step (in millions),
    starting at 0M.
    """
    import re

    import matplotlib
    import numpy as _np

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection, PatchCollection
    from matplotlib.patches import FancyArrowPatch, Rectangle

    files = []
    for name in os.listdir(save_dir):
        match = re.match(r"^(\d+)_agent_positions\.npz$", name)
        if match:
            files.append((int(match.group(1)), os.path.join(save_dir, name)))

    if not files:
        raise FileNotFoundError(f"No '*_agent_positions.npz' files found in {save_dir}")

    files.sort(key=lambda x: x[0])
    ends = _early_dense_indices(len(files), n_snapshots, power=early_spacing_power)

    if num_steps is not None and num_envs is not None:
        env_steps_per_update = int(num_steps) * int(num_envs)
    else:
        env_steps_per_update = _load_env_steps_per_update(save_dir)

    windows = []
    last_env_steps_m = []
    for i, end_i in enumerate(ends):
        start_i = 0 if i == 0 else int(ends[i - 1]) + 1
        end_i = int(end_i)
        # First panel is only the initial checkpoint so its label is 0M.
        if i == 0:
            start_i = 0
            end_i = 0
        xy_parts = []
        for file_i in range(start_i, end_i + 1):
            with _np.load(files[file_i][1]) as data:
                xy_parts.append(_np.asarray(data["agent_xy"]).reshape(-1, 2))
        windows.append(
            _np.concatenate(xy_parts, axis=0)
            if xy_parts
            else _np.zeros((0, 2), dtype=_np.float64)
        )
        snap_update = int(files[end_i][0])
        last_env_steps_m.append(snap_update * env_steps_per_update / 1e6)

    segments, fills, starts, goals, xlim, ylim = maze_wall_segments(maze_name)
    wall_patches = [Rectangle((x0, y0), w, h) for x0, y0, w, h in fills]

    fig, axes = plt.subplots(
        1,
        len(windows),
        figsize=(3.2 * len(windows), 4.2),
        gridspec_kw={"wspace": 0.08},
    )
    if len(windows) == 1:
        axes = [axes]

    for ax, xy in zip(axes, windows):
        if wall_patches:
            ax.add_collection(
                PatchCollection(
                    wall_patches,
                    facecolor="#d0d0d0",
                    edgecolor="none",
                    zorder=1,
                    rasterized=True,
                )
            )
        if xy.shape[0] > 0:
            ax.scatter(
                xy[:, 0],
                xy[:, 1],
                c="#2171b5",
                s=4,
                alpha=0.7,
                linewidths=0,
                zorder=2,
                rasterized=True,
            )
        if segments.shape[0] > 0:
            ax.add_collection(
                LineCollection(
                    segments,
                    colors="black",
                    linewidths=1.2,
                    zorder=3,
                    rasterized=True,
                )
            )
        if starts.shape[0] > 0:
            ax.scatter(
                starts[:, 0],
                starts[:, 1],
                c="#2ca02c",
                s=55,
                marker="o",
                edgecolors="black",
                linewidths=0.8,
                zorder=4,
            )
        if goals.shape[0] > 0:
            ax.scatter(
                goals[:, 0],
                goals[:, 1],
                c="#d62728",
                s=55,
                marker="*",
                edgecolors="black",
                linewidths=0.6,
                zorder=4,
            )
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect("equal")
        ax.axis("off")
        # Rasterize dense scatter/wall layers in PDF; keep markers/text vector.
        ax.set_rasterization_zorder(3.5)

    title_name = maze_name
    for suffix in ("_single_goal", "_all_goals", "_eval"):
        if title_name.endswith(suffix):
            title_name = title_name[: -len(suffix)]
            break
    title_parts = title_name.split("_")
    title = f"{title_parts[0].capitalize()}-{'-'.join(title_parts[1:])} coverage"
    title_fs = 18 * font_scale
    tick_fs = 13 * font_scale
    label_fs = 14 * font_scale
    fig.suptitle(title, fontsize=title_fs, y=0.98)
    fig.subplots_adjust(left=0.04, right=0.96, top=0.86, bottom=0.28)

    fig.canvas.draw()
    first_bbox = axes[0].get_position()
    last_bbox = axes[-1].get_position()
    arrow_y = first_bbox.y0 - 0.07
    x_left = first_bbox.x0
    x_right = last_bbox.x0 + last_bbox.width

    arrow = FancyArrowPatch(
        (x_left, arrow_y),
        (x_right, arrow_y),
        transform=fig.transFigure,
        arrowstyle="<->",
        mutation_scale=14 * font_scale,
        linewidth=1.4,
        color="black",
        clip_on=False,
    )
    fig.add_artist(arrow)

    for ax, steps_m in zip(axes, last_env_steps_m):
        bbox = ax.get_position()
        x_center = bbox.x0 + 0.5 * bbox.width
        if float(steps_m).is_integer() or abs(steps_m - round(steps_m)) < 1e-6:
            label = f"{int(round(steps_m))}M"
        else:
            label = f"{steps_m:.1f}M"
        # Tick mark from arrow up toward the panel.
        fig.add_artist(
            plt.Line2D(
                [x_center, x_center],
                [arrow_y - 0.008, arrow_y + 0.012],
                transform=fig.transFigure,
                color="black",
                linewidth=1.2,
                clip_on=False,
            )
        )
        fig.text(
            x_center,
            arrow_y - 0.04,
            label,
            ha="center",
            va="top",
            fontsize=tick_fs,
            transform=fig.transFigure,
        )

    fig.text(
        0.5 * (x_left + x_right),
        arrow_y - 0.125,
        "Environment steps",
        ha="center",
        va="top",
        fontsize=label_fs,
        transform=fig.transFigure,
    )

    if output_path is None:
        output_path = os.path.join(save_dir, "coverage_snapshots.png")
    output_path_pdf = os.path.splitext(output_path)[0] + ".pdf"

    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    # dpi controls rasterized artist resolution inside the PDF.
    fig.savefig(output_path_pdf, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved coverage snapshots to {output_path}")
    print(f"Saved coverage snapshots pdf to {output_path_pdf}")
    return output_path


def _maze_snapshot_title(maze_name, kind="coverage"):
    title_name = maze_name
    for suffix in ("_single_goal", "_all_goals", "_eval"):
        if title_name.endswith(suffix):
            title_name = title_name[: -len(suffix)]
            break
    title_parts = title_name.split("_")
    pretty = f"{title_parts[0].capitalize()}-{'-'.join(title_parts[1:])}"
    return f"{pretty} {kind}"


def _add_env_steps_arrow(fig, axes, last_env_steps_m, font_scale=1.0):
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyArrowPatch

    tick_fs = 13 * font_scale
    label_fs = 14 * font_scale

    fig.canvas.draw()
    first_bbox = axes[0].get_position()
    last_bbox = axes[-1].get_position()
    arrow_y = first_bbox.y0 - 0.07
    x_left = first_bbox.x0
    x_right = last_bbox.x0 + last_bbox.width

    fig.add_artist(
        FancyArrowPatch(
            (x_left, arrow_y),
            (x_right, arrow_y),
            transform=fig.transFigure,
            arrowstyle="<->",
            mutation_scale=14 * font_scale,
            linewidth=1.4,
            color="black",
            clip_on=False,
        )
    )

    for ax, steps_m in zip(axes, last_env_steps_m):
        bbox = ax.get_position()
        x_center = bbox.x0 + 0.5 * bbox.width
        if float(steps_m).is_integer() or abs(steps_m - round(steps_m)) < 1e-6:
            label = f"{int(round(steps_m))}M"
        else:
            label = f"{steps_m:.1f}M"
        fig.add_artist(
            plt.Line2D(
                [x_center, x_center],
                [arrow_y - 0.008, arrow_y + 0.012],
                transform=fig.transFigure,
                color="black",
                linewidth=1.2,
                clip_on=False,
            )
        )
        fig.text(
            x_center,
            arrow_y - 0.04,
            label,
            ha="center",
            va="top",
            fontsize=tick_fs,
            transform=fig.transFigure,
        )

    fig.text(
        0.5 * (x_left + x_right),
        arrow_y - 0.125,
        "Environment steps",
        ha="center",
        va="top",
        fontsize=label_fs,
        transform=fig.transFigure,
    )


def plot_teacher_goal_snapshots(
    save_dir,
    maze_name,
    output_path=None,
    n_snapshots=8,
    cmap="Blues",
    dpi=150,
    early_spacing_power=2.0,
    font_scale=1.0,
):
    """Teacher goal-selection heatmaps across training, coverage-style layout.

    Prefers ``*_teacher_goal_counts_{step}.npz`` (counts + goal grid). Falls
    back to the saved PNG heatmaps when npz files are unavailable. Draws the
    same maze frame / start / goals overlay as coverage when using npz data.
    """
    import re

    import matplotlib
    import numpy as _np

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection, PatchCollection
    from matplotlib.patches import Rectangle

    npz_files = []
    png_files = []
    for name in os.listdir(save_dir):
        match_npz = re.match(r"^.*_teacher_goal_counts_(\d+)\.npz$", name)
        if match_npz:
            npz_files.append((int(match_npz.group(1)), os.path.join(save_dir, name)))
            continue
        match_png = re.match(r"^.*_teacher_goal_counts_(\d+)\.png$", name)
        if match_png:
            png_files.append((int(match_png.group(1)), os.path.join(save_dir, name)))

    use_npz = len(npz_files) > 0
    files = sorted(npz_files if use_npz else png_files, key=lambda x: x[0])
    if not files:
        raise FileNotFoundError(
            f"No '*_teacher_goal_counts_*.npz' or '*.png' files found in {save_dir}"
        )

    pick = _early_dense_indices(len(files), n_snapshots, power=early_spacing_power)
    selected = [files[int(i)] for i in pick]

    last_env_steps_m = [step / 1e6 for step, _ in selected]

    fig, axes = plt.subplots(
        1,
        len(selected),
        figsize=(3.2 * len(selected), 4.2),
        gridspec_kw={"wspace": 0.08},
    )
    if len(selected) == 1:
        axes = [axes]

    if use_npz:
        segments, fills, starts, goals, xlim, ylim = maze_wall_segments(maze_name)
        wall_patches = [Rectangle((x0, y0), w, h) for x0, y0, w, h in fills]

        for ax, (_, fname) in zip(axes, selected):
            with _np.load(fname) as data:
                counts = _np.asarray(data["counts"]).reshape(-1)
                goal_grid_xy = _np.asarray(data["goal_grid_xy"])
                num_points = int(data["num_points"])

            gx = goal_grid_xy[:, 0].reshape(num_points, num_points)
            gy = goal_grid_xy[:, 1].reshape(num_points, num_points)
            cgrid = counts.reshape(num_points, num_points).astype(_np.float32)
            max_count = float(cgrid.max()) if cgrid.size else 0.0
            normalized = (
                cgrid / max_count if max_count > 0.0 else _np.zeros_like(cgrid)
            )

            ax.pcolormesh(
                gx,
                gy,
                normalized,
                shading="nearest",
                cmap=cmap,
                vmin=0.0,
                vmax=1.0,
                zorder=0,
                rasterized=True,
            )
            if wall_patches:
                ax.add_collection(
                    PatchCollection(
                        wall_patches,
                        facecolor="#d0d0d0",
                        edgecolor="none",
                        zorder=1,
                        rasterized=True,
                    )
                )
            if segments.shape[0] > 0:
                ax.add_collection(
                    LineCollection(
                        segments,
                        colors="black",
                        linewidths=1.2,
                        zorder=3,
                        rasterized=True,
                    )
                )
            if starts.shape[0] > 0:
                ax.scatter(
                    starts[:, 0],
                    starts[:, 1],
                    c="#2ca02c",
                    s=55,
                    marker="o",
                    edgecolors="black",
                    linewidths=0.8,
                    zorder=4,
                )
            if goals.shape[0] > 0:
                ax.scatter(
                    goals[:, 0],
                    goals[:, 1],
                    c="#d62728",
                    s=55,
                    marker="*",
                    edgecolors="black",
                    linewidths=0.6,
                    zorder=4,
                )
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            ax.set_aspect("equal")
            ax.axis("off")
            ax.set_rasterization_zorder(3.5)
    else:
        for ax, (_, fname) in zip(axes, selected):
            img = plt.imread(fname)
            ax.imshow(img)
            ax.axis("off")

    fig.suptitle(
        _maze_snapshot_title(maze_name, "teacher goal distribution"),
        fontsize=18 * font_scale,
        y=0.98,
    )
    fig.subplots_adjust(left=0.04, right=0.96, top=0.86, bottom=0.28)
    _add_env_steps_arrow(fig, axes, last_env_steps_m, font_scale=font_scale)

    if output_path is None:
        output_path = os.path.join(save_dir, "teacher_goal_snapshots.png")
    output_path_pdf = os.path.splitext(output_path)[0] + ".pdf"

    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(output_path_pdf, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved teacher goal snapshots to {output_path}")
    return output_path


def _evaluation_goal_structure(maze_module, layout_name):
    """Layout used for evaluation / all-possible goals for ``layout_name``."""
    if "u_maze" in layout_name:
        return maze_module.U_MAZE_ALL_STATES
    if "big_maze" in layout_name:
        return maze_module.BIG_MAZE_ALL_GOALS
    if "hardest" in layout_name and hasattr(maze_module, "HARDEST_MAZE"):
        return maze_module.HARDEST_MAZE
    raise ValueError(
        f"No evaluation-goal layout for maze layout {layout_name!r}"
    )


def _maze_pretty_name(maze_name):
    title_name = maze_name
    for suffix in ("_single_goal", "_all_goals", "_eval"):
        if title_name.endswith(suffix):
            title_name = title_name[: -len(suffix)]
            break
    title_parts = title_name.split("_")
    return f"{title_parts[0].capitalize()}-{'-'.join(title_parts[1:])}"


def _evaluation_goals_xy(maze_name):
    """Return ``(goals, segments, fills, starts, xlim, ylim)`` for ``maze_name``."""
    import inspect

    import numpy as _np
    from matplotlib.patches import Rectangle

    maze_module, env_cls, layout_name = _maze_env_for_name(maze_name)
    scale = float(
        inspect.signature(env_cls.__init__).parameters["maze_size_scaling"].default
    )
    structure = _evaluation_goal_structure(maze_module, layout_name)
    goal_token = maze_module.GOAL
    goals = _np.asarray(
        [
            [i * scale, j * scale]
            for i, row in enumerate(structure)
            for j, cell in enumerate(row)
            if cell == goal_token
        ],
        dtype=_np.float64,
    )
    if goals.size == 0:
        goals = _np.zeros((0, 2), dtype=_np.float64)

    segments, fills, starts, _, xlim, ylim = maze_wall_segments(maze_name)
    return goals, segments, fills, starts, xlim, ylim


def _draw_evaluation_goals_ax(ax, maze_name):
    """Draw coverage-style maze + yellow evaluation goals on ``ax``."""
    from matplotlib.collections import LineCollection, PatchCollection
    from matplotlib.patches import Rectangle

    goals, segments, fills, starts, xlim, ylim = _evaluation_goals_xy(maze_name)
    wall_patches = [Rectangle((x0, y0), w, h) for x0, y0, w, h in fills]

    if wall_patches:
        ax.add_collection(
            PatchCollection(
                wall_patches,
                facecolor="#d0d0d0",
                edgecolor="none",
                zorder=1,
                rasterized=True,
            )
        )
    if segments.shape[0] > 0:
        ax.add_collection(
            LineCollection(
                segments,
                colors="black",
                linewidths=1.2,
                zorder=3,
                rasterized=True,
            )
        )
    if starts.shape[0] > 0:
        ax.scatter(
            starts[:, 0],
            starts[:, 1],
            c="#2ca02c",
            s=70,
            marker="o",
            edgecolors="black",
            linewidths=0.8,
            zorder=4,
        )
    if goals.shape[0] > 0:
        ax.scatter(
            goals[:, 0],
            goals[:, 1],
            c="#ffd700",
            s=70,
            marker="o",
            edgecolors="black",
            linewidths=0.6,
            zorder=5,
            rasterized=True,
        )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_rasterization_zorder(3.5)


def _draw_maze_layout_ax(ax, maze_name):
    """Draw coverage-style maze walls + layout start/goals on ``ax``.

    Walls and outer frame match ``plot_coverage_snapshots``. Starts (``R``)
    and goals (``G``) come from the env layout via ``maze_wall_segments``,
    not the evaluation all-goals layout.
    """
    from matplotlib.collections import LineCollection, PatchCollection
    from matplotlib.patches import Rectangle

    segments, fills, starts, goals, xlim, ylim = maze_wall_segments(maze_name)
    wall_patches = [Rectangle((x0, y0), w, h) for x0, y0, w, h in fills]

    if wall_patches:
        ax.add_collection(
            PatchCollection(
                wall_patches,
                facecolor="#d0d0d0",
                edgecolor="none",
                zorder=1,
                rasterized=True,
            )
        )
    if segments.shape[0] > 0:
        ax.add_collection(
            LineCollection(
                segments,
                colors="black",
                linewidths=1.2,
                zorder=3,
                rasterized=True,
            )
        )
    if starts.shape[0] > 0:
        ax.scatter(
            starts[:, 0],
            starts[:, 1],
            c="#2ca02c",
            s=55,
            marker="o",
            edgecolors="black",
            linewidths=0.8,
            zorder=4,
        )
    if goals.shape[0] > 0:
        ax.scatter(
            goals[:, 0],
            goals[:, 1],
            c="#d62728",
            s=55,
            marker="*",
            edgecolors="black",
            linewidths=0.6,
            zorder=4,
        )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_rasterization_zorder(3.5)


def plot_maze_layout(
    maze_name,
    output_path=None,
    dpi=250,
    font_scale=1.0,
):
    """Plot maze walls and layout start/goals in coverage style.

    Gray walls and black outer frame match ``plot_coverage_snapshots``.
    Starts (green) and goals (red stars) are the ``R`` / ``G`` cells from
    the env layout only (no evaluation all-goals overlay).
    """
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    _draw_maze_layout_ax(ax, maze_name)

    fig.subplots_adjust(left=0.04, right=0.96, top=0.96, bottom=0.04)

    if output_path is None:
        output_path = f"{maze_name}_maze_layout.png"
    output_path_pdf = os.path.splitext(output_path)[0] + ".pdf"

    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(output_path_pdf, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved maze layout to {output_path}")
    return output_path


def plot_evaluation_goals(
    maze_name,
    output_path=None,
    dpi=250,
    font_scale=1.0,
):
    """Plot all evaluation goals as yellow points on the coverage-style maze.

    Walls, outer frame, and start markers match ``plot_coverage_snapshots``.
    Evaluation goals come from the maze's all-goals / all-states layout
    (same family as training ``all_possible_goals``).
    """
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    _draw_evaluation_goals_ax(ax, maze_name)

    pretty = _maze_pretty_name(maze_name)
    fig.suptitle(
        f"{pretty}-evaluation goals",
        fontsize=18 * font_scale,
        y=0.98,
    )
    fig.subplots_adjust(left=0.04, right=0.96, top=0.90, bottom=0.04)

    if output_path is None:
        output_path = f"{maze_name}_evaluation_goals.png"
    output_path_pdf = os.path.splitext(output_path)[0] + ".pdf"

    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(output_path_pdf, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved evaluation goals to {output_path}")
    return output_path


def plot_evaluation_goals_row(
    maze_names,
    output_path=None,
    dpi=250,
    font_scale=1.0,
):
    """Plot evaluation goals for several mazes in one row.

    Each panel uses the same coverage-style maze rendering as
    ``plot_evaluation_goals``, with title ``{pretty}-evaluation goals``.
    """
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    maze_names = list(maze_names)
    if not maze_names:
        raise ValueError("maze_names must contain at least one maze")

    n = len(maze_names)
    fig, axes = plt.subplots(
        1,
        n,
        figsize=(5.2 * n, 5.5),
        gridspec_kw={"wspace": 0.12},
    )
    if n == 1:
        axes = [axes]

    for ax, maze_name in zip(axes, maze_names):
        _draw_evaluation_goals_ax(ax, maze_name)
        pretty = _maze_pretty_name(maze_name)
        ax.set_title(
            f"{pretty}-evaluation goals",
            fontsize=16 * font_scale,
            pad=10,
        )

    fig.subplots_adjust(left=0.02, right=0.98, top=0.88, bottom=0.04)

    if output_path is None:
        slug = "_".join(_maze_pretty_name(m).replace("-", "_") for m in maze_names)
        output_path = f"{slug}_evaluation_goals_row.png"
    output_path_pdf = os.path.splitext(output_path)[0] + ".pdf"

    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(output_path_pdf, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved evaluation goals row to {output_path}")
    return output_path


In [3]:
exp_path = "/network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/teapot_42374677"
positions_path = os.path.join(exp_path, "agent_positions")
goal_counts_path = os.path.join(exp_path, "teacher_goal_visuals")
empowerment_path = os.path.join(exp_path, "teacher_empowerment_visuals")
lp_path = os.path.join(exp_path, "lp_reward_visual")
start = 0
end = -1
output = load_agent_positions(positions_path, start, end)
xy = output["agent_xy"]
import matplotlib.pyplot as plt
plt.figure()
plt.scatter(xy[:, 0], xy[:, 1], s=1, alpha=0.1)
# plt.axis("off")
# plt.show()
# plt.close()

NameError: name 'os' is not defined

In [7]:
plot_maze_layout("ant_u_maze_single_goal")

Saved maze layout to ant_u_maze_single_goal_maze_layout.png


'ant_u_maze_single_goal_maze_layout.png'

In [43]:
coverage_path = plot_coverage_snapshots(
    positions_path,
    maze_name="ant_big_maze_single_goal",
    n_snapshots=6,
    dpi=500,
    font_scale=1.3,
)


Saved coverage snapshots to /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/teapot_42374677/agent_positions/coverage_snapshots.png
Saved coverage snapshots pdf to /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/teapot_42374677/agent_positions/coverage_snapshots.pdf


In [44]:
# eval_goals_path = plot_evaluation_goals(
#     maze_name="ant_big_maze_single_goal",
#     output_path=os.path.join(exp_path, "evaluation_goals.png"),
#     dpi=500,
#     font_scale=1.0,
# )

eval_goals_row_path = plot_evaluation_goals_row(
    maze_names=[
        "ant_u_maze_single_goal",
        "ant_big_maze_single_goal",
    ],
    output_path=os.path.join(exp_path, "evaluation_goals_row.png"),
    dpi=500,
    font_scale=1.0,
)


Saved evaluation goals to /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/teapot_42374677/evaluation_goals.png
Saved evaluation goals row to /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/teapot_42374677/evaluation_goals_row.png


In [34]:
teacher_goal_path = plot_teacher_goal_snapshots(
    goal_counts_path,
    maze_name="ant_big_maze_single_goal",
    n_snapshots=3,
)


Saved teacher goal snapshots to /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/teapot_42374677/teacher_goal_visuals/teacher_goal_snapshots.png


In [15]:
output = load_teacher_goal_visuals(goal_counts_path, start, end, duration=0.5)

FileNotFoundError: No '*_teacher_goal_counts_*.png' files found in /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/teapot_42374677/teacher_goal_visuals

In [4]:
output = load_empowerment_visuals(empowerment_path, start, end, duration=0.5)

FileNotFoundError: [Errno 2] No such file or directory: '/network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/recommend_138013809/teacher_empowerment_visuals'

In [ ]:
end =-1
output = load_agent_positions_visuals(positions_path, start, end, duration=0.1, render_mode="scatter")

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (785, 754) to (800, 768) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Saved agent positions visuals to /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/recommend_138013809/agent_positions/agent_positions_visuals_0_-1.mp4


In [ ]:
load_lp_visuals(lp_path, start, end, duration=0.5)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (959, 817) to (960, 832) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


Saved teacher lp visuals to /network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/recommend_138013809/lp_reward_visual/teacher_lp_visuals_0_-1.mp4


{'video_path': '/network/scratch/f/faisal.mohamed/purejaxrl_simple_teachers/recommend_138013809/lp_reward_visual/teacher_lp_visuals_0_-1.mp4',
 'update_idx': [0,
  8192000,
  16384000,
  24576000,
  32768000,
  40960000,
  49152000,
  57344000,
  65536000,
  73728000,
  81920000,
  90112000,
  98304000,
  106496000,
  114688000,
  122880000,
  131072000,
  139264000,
  147456000,
  155648000,
  163840000,
  172032000,
  180224000,
  188416000,
  196608000,
  204800000,
  212992000,
  221184000,
  229376000,
  237568000,
  245760000,
  253952000,
  262144000,
  270336000,
  278528000,
  286720000],
 'frames': [array([[[255, 255, 255, 255],
          [255, 255, 255, 255],
          [255, 255, 255, 255],
          ...,
          [255, 255, 255, 255],
          [255, 255, 255, 255],
          [255, 255, 255, 255]],
  
         [[255, 255, 255, 255],
          [255, 255, 255, 255],
          [255, 255, 255, 255],
          ...,
          [255, 255, 255, 255],
          [255, 255, 255, 255],